# Naïve vs. Hybrid RAG — Comparative Evaluation

This notebook runs both retrieval strategies on the same test set and compares them on four metrics:

- **Context Precision @ 3** — % of retrieved chunks that match expected sources
- **Recall @ 3** — % of expected sources captured in top-3
- **Citation Coverage** — % of citations in the answer that map to retrieved chunks (groundedness proxy)
- **% with Freshness Flag** — does the answer surface time-sensitivity?

This is a *methodology* proof for the eventual three-way comparison (Naïve vs. CRAG vs. Self-RAG) that the project's research contribution requires.

**Hypothesis going in:** Hybrid should outperform Naïve on Context Precision and Recall because BM25 catches keyword matches the embedding misses (e.g., 'IRAP' as an acronym), and the union before fusion increases recall. Citation Coverage and Freshness Flag depend on the generator, not the retriever, so should be similar across modes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

from retriever import Retriever
from evaluator import evaluate_mode, TEST_SET
import pandas as pd

In [ ]:
r = Retriever()
naive = evaluate_mode(r.retrieve_naive, 'naive')
hybrid = evaluate_mode(r.retrieve_hybrid, 'hybrid')

In [ ]:
summary = pd.DataFrame([
    {
        'metric': 'Context Precision @ 3',
        'naive': naive['avg_context_precision'],
        'hybrid': hybrid['avg_context_precision'],
    },
    {
        'metric': 'Recall @ 3',
        'naive': naive['avg_recall_at_3'],
        'hybrid': hybrid['avg_recall_at_3'],
    },
    {
        'metric': 'Citation Coverage',
        'naive': naive['avg_citation_coverage'],
        'hybrid': hybrid['avg_citation_coverage'],
    },
    {
        'metric': '% w/ Freshness Flag',
        'naive': naive['pct_with_freshness_flag'],
        'hybrid': hybrid['pct_with_freshness_flag'],
    },
])
summary['delta'] = summary['hybrid'] - summary['naive']
summary.style.format({'naive': '{:.1%}', 'hybrid': '{:.1%}', 'delta': '{:+.1%}'})

## Per-query breakdown

Useful for spotting cases where Naïve and Hybrid disagree — those are the most informative cases to inspect manually.

In [ ]:
per_query = pd.DataFrame([
    {
        'query': n['query'][:60] + '...' if len(n['query']) > 60 else n['query'],
        'expected': ', '.join(n['expected_ids']),
        'naive_retrieved': ', '.join(n['retrieved_ids']),
        'hybrid_retrieved': ', '.join(h['retrieved_ids']),
        'naive_precision': n['context_precision'],
        'hybrid_precision': h['context_precision'],
    }
    for n, h in zip(naive['rows'], hybrid['rows'])
])
per_query

## Reading the results

**If Hybrid > Naïve on Context Precision:** BM25 is catching keyword signals (e.g., organization acronyms like 'IRAP', 'CCRM') that pure embedding similarity misses.

**If Naïve > Hybrid on some queries:** Likely a query where the semantic similarity is doing most of the work and BM25 is adding noise. This is expected on natural-language queries with no rare keywords. It's also why the production design would adaptively weight the two retrievers per query type rather than using fixed RRF.

**If both are low:** The seed data is probably too sparse to answer that query type. In production this signals a gap in the index — a candidate for the CRAG web-fallback path.

## Next steps in the real project

1. Build a larger test set (~50 queries) authored from H2i mentor logs and founder interviews.
2. Replace the proxy metrics here with full RAGAs metrics (faithfulness, answer_relevance, context_precision, context_recall).
3. Add a 'stale data probe' set — queries about programs whose status changed between index build and query — to measure how each architecture handles staleness.
4. Run a qualitative end-user satisfaction survey alongside the quantitative metrics. Per Zhang's NEJM AI paper, RAGAs alone doesn't capture domain complexity.